# 🎓 캡스톤 프로젝트 템플릿

**30일 딥러닝 완전 정복 · Day 29–30**

이 노트북은 빈칸을 채워 가며 프로젝트를 진행하는 틀입니다. `TODO`라고 적힌 곳을 여러분의 문제에 맞게 고치세요.
강의 Day 29의 체크리스트와 Day 28의 디버깅 레시피를 옆에 띄워 두세요.

> ⚙️ 런타임 → 런타임 유형 변경 → T4 GPU

## 0. 문제 정의서 (여기에 직접 작성)
1. **한 문장 목표**: TODO
2. **입력**: TODO (형식, 텐서 형상)
3. **출력 / 출력층 / 손실 함수**: TODO
4. **평가 지표와 이유**: TODO
5. **성공 기준**: TODO
6. **분할 계획**: TODO
7. **예상되는 어려움**: TODO

## 1. 공통 템플릿 (Day 16)

In [ ]:
import random, os, json, time
import numpy as np, torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import DataLoader, random_split
from torchvision import datasets, transforms
from tqdm.auto import tqdm
device = "cuda" if torch.cuda.is_available() else "cpu"

def seed_everything(seed=42):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)

def train_one_epoch(model, loader, loss_fn, opt, scheduler=None, clip=None, scaler=None):
    model.train(); tot, correct, n = 0.0, 0, 0
    for xb, yb in tqdm(loader, leave=False):
        xb, yb = xb.to(device, non_blocking=True), yb.to(device, non_blocking=True)
        with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=scaler is not None):
            logits = model(xb); loss = loss_fn(logits, yb)
        opt.zero_grad(set_to_none=True)
        if scaler is not None:
            scaler.scale(loss).backward()
            if clip: scaler.unscale_(opt); torch.nn.utils.clip_grad_norm_(model.parameters(), clip)
            scaler.step(opt); scaler.update()
        else:
            loss.backward()
            if clip: torch.nn.utils.clip_grad_norm_(model.parameters(), clip)
            opt.step()
        if scheduler is not None: scheduler.step()
        tot += loss.item() * len(yb); correct += (logits.argmax(1) == yb).sum().item(); n += len(yb)
    return tot / n, correct / n

@torch.no_grad()
def evaluate(model, loader, loss_fn):
    model.eval(); tot, correct, n = 0.0, 0, 0
    for xb, yb in loader:
        xb, yb = xb.to(device), yb.to(device)
        logits = model(xb); tot += loss_fn(logits, yb).item() * len(yb)
        correct += (logits.argmax(1) == yb).sum().item(); n += len(yb)
    return tot / n, correct / n

def save_ckpt(path, model, opt, epoch, best):
    torch.save({"model": model.state_dict(), "opt": opt.state_dict(), "epoch": epoch, "best": best}, path)

def load_ckpt(path, model, opt=None):
    ck = torch.load(path, map_location=device)
    model.load_state_dict(ck["model"])
    if opt is not None: opt.load_state_dict(ck["opt"])
    return ck["epoch"], ck["best"]

def fit(model, train_dl, val_dl, epochs, lr=1e-3, wd=0.05, ckpt="best.pt", patience=None, label_smoothing=0.0, amp=True, sgd=False):
    loss_fn = nn.CrossEntropyLoss(label_smoothing=label_smoothing)
    if sgd: opt = torch.optim.SGD(model.parameters(), lr=lr, momentum=0.9, nesterov=True, weight_decay=wd)
    else:   opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=wd)
    sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=lr, total_steps=epochs * len(train_dl), pct_start=0.15)
    scaler = torch.amp.GradScaler("cuda") if (amp and device == "cuda") else None
    hist = {k: [] for k in ["tr_loss", "tr_acc", "va_loss", "va_acc"]}; best, bad = 0.0, 0
    for ep in range(epochs):
        t0 = time.time()
        tl, ta = train_one_epoch(model, train_dl, loss_fn, opt, sched, clip=1.0, scaler=scaler)
        vl, va = evaluate(model, val_dl, nn.CrossEntropyLoss())
        for k, v in zip(hist, [tl, ta, vl, va]): hist[k].append(v)
        print(f"[{ep+1:02d}/{epochs}] train {tl:.4f}/{ta:.3f}  val {vl:.4f}/{va:.3f}  lr {opt.param_groups[0]['lr']:.2e}  {time.time()-t0:.0f}s")
        if va > best: best, bad = va, 0; save_ckpt(ckpt, model, opt, ep, best)
        else:
            bad += 1
            if patience and bad >= patience: print("early stopping"); break
    load_ckpt(ckpt, model)
    return hist

def plot_hist(hist, title=""):
    import matplotlib.pyplot as plt
    fig, ax = plt.subplots(1, 2, figsize=(12, 3.5))
    ax[0].plot(hist["tr_loss"], label="train"); ax[0].plot(hist["va_loss"], label="val"); ax[0].set_title(f"{title} loss"); ax[0].legend()
    ax[1].plot(hist["tr_acc"], label="train"); ax[1].plot(hist["va_acc"], label="val"); ax[1].set_title(f"{title} accuracy"); ax[1].legend()
    plt.show()
print("template ready on", device)

## 2. 데이터
아래는 이미지 폴더(`root/클래스명/*.jpg`) 예시입니다. 다른 유형이면 Day 16의 `Dataset` 예시를 참고해 바꾸세요.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
DATA_ROOT = "/content/drive/MyDrive/capstone_data"      # TODO: 데이터 경로
IMG = 224                                               # TODO: 입력 크기
mean, std = (0.485, 0.456, 0.406), (0.229, 0.224, 0.225)
train_tf = transforms.Compose([transforms.RandomResizedCrop(IMG, scale=(0.6, 1)), transforms.RandomHorizontalFlip(),   # TODO: 도메인에 맞는 증강인가?
                               transforms.ToTensor(), transforms.Normalize(mean, std)])
eval_tf = transforms.Compose([transforms.Resize(int(IMG * 1.14)), transforms.CenterCrop(IMG), transforms.ToTensor(), transforms.Normalize(mean, std)])

full_tr = datasets.ImageFolder(DATA_ROOT, transform=train_tf)
full_ev = datasets.ImageFolder(DATA_ROOT, transform=eval_tf)
classes = full_tr.classes; N = len(full_tr)
g = torch.Generator().manual_seed(0); perm = torch.randperm(N, generator=g)
n_te, n_va = int(0.15 * N), int(0.15 * N)
te_idx, va_idx, tr_idx = perm[:n_te], perm[n_te:n_te + n_va], perm[n_te + n_va:]   # 테스트는 마지막까지 봉인!
train_dl = DataLoader(torch.utils.data.Subset(full_tr, tr_idx), 32, shuffle=True, num_workers=2, drop_last=True)
val_dl = DataLoader(torch.utils.data.Subset(full_ev, va_idx), 64, num_workers=2)
test_dl = DataLoader(torch.utils.data.Subset(full_ev, te_idx), 64, num_workers=2)
print(classes, len(tr_idx), len(va_idx), len(te_idx))

## 3. 데이터 점검 (Day 28 레시피 1단계)

In [ ]:
import matplotlib.pyplot as plt
xb, yb = next(iter(train_dl))
print("x:", xb.shape, xb.dtype, f"min {xb.min():.2f} max {xb.max():.2f}"); print("y:", yb[:16].tolist())
print("class counts:", torch.bincount(torch.tensor([full_tr.targets[i] for i in tr_idx])).tolist())
fig, axs = plt.subplots(2, 8, figsize=(16, 4.5))
for a, x, y in zip(axs.ravel(), xb, yb):
    a.imshow((x.permute(1, 2, 0) * torch.tensor(std) + torch.tensor(mean)).clamp(0, 1)); a.set_title(classes[y], fontsize=8); a.axis("off")
plt.show()

## 4. 베이스라인 사다리

In [ ]:
results = {}
# (1) 바보 베이스라인: 최빈 클래스
counts = torch.bincount(torch.tensor([full_tr.targets[i] for i in tr_idx]))
majority = counts.argmax().item()
val_targets = torch.tensor([full_tr.targets[i] for i in va_idx])
results["majority"] = (val_targets == majority).float().mean().item()
print(results)

In [ ]:
# (2) 사전학습 특징 추출 (Day 19)
from torchvision import models
def make_model(freeze=True):
    m = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V2)      # TODO: 다른 백본?
    if freeze:
        for p in m.parameters(): p.requires_grad = False
    m.fc = nn.Sequential(nn.Dropout(0.2), nn.Linear(m.fc.in_features, len(classes)))
    return m.to(device)
seed_everything(0)
model = make_model()
# 배치 하나 과적합 테스트
xb, yb = next(iter(train_dl)); xb, yb = xb[:16].to(device), yb[:16].to(device)
o = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad], 1e-2)
for i in range(100): l = F.cross_entropy(model(xb), yb); o.zero_grad(); l.backward(); o.step()
print("single-batch loss:", l.item())

In [ ]:
seed_everything(0)
model = make_model(freeze=True)
hist = fit(model, train_dl, val_dl, epochs=10, lr=3e-3, ckpt="cap_fe.pt")
plot_hist(hist, "feature extraction"); results["feature_extraction"] = max(hist["va_acc"]); print(results)

## 5. 실험 기록표
| # | 가설 | 변경 | val 지표 | 결론 |
|---|---|---|---|---|
| 0 | 베이스라인 |  |  |  |
| 1 | TODO |  |  |  |

In [ ]:
# (3) 미세조정 실험 (Day 19)  — TODO: 여러분의 실험을 여기에 하나씩 추가
for p in model.layer4.parameters(): p.requires_grad = True
opt = torch.optim.AdamW([{"params": model.layer4.parameters(), "lr": 1e-4}, {"params": model.fc.parameters(), "lr": 1e-3}], weight_decay=0.01)
best = 0
for ep in range(8):
    tl, ta = train_one_epoch(model, train_dl, nn.CrossEntropyLoss(label_smoothing=0.1), opt)
    vl, va = evaluate(model, val_dl, nn.CrossEntropyLoss())
    print(f"[{ep+1}] train {tl:.3f}/{ta:.3f}  val {vl:.3f}/{va:.3f}")
    if va > best: best = va; torch.save(model.state_dict(), "cap_ft.pt")
results["fine_tune"] = best; model.load_state_dict(torch.load("cap_ft.pt")); print(results)

## 6. 최종 평가 (테스트 세트 — 단 한 번!)

In [ ]:
test_loss, test_acc = evaluate(model, test_dl, nn.CrossEntropyLoss())
print(f"TEST accuracy {test_acc:.4f}")
model.eval(); P, Y, X = [], [], []
with torch.no_grad():
    for xb, yb in test_dl: P.append(F.softmax(model(xb.to(device)), -1).cpu()); Y.append(yb); X.append(xb)
P, Y, X = torch.cat(P), torch.cat(Y), torch.cat(X); pred = P.argmax(1)
K = len(classes); cm = torch.zeros(K, K, dtype=torch.long)
for t, p in zip(Y, pred): cm[t, p] += 1
plt.figure(figsize=(6, 5)); plt.imshow(cm, cmap="Blues"); plt.xticks(range(K), classes, rotation=60); plt.yticks(range(K), classes); plt.colorbar(); plt.show()
wrong = torch.where(pred != Y)[0]; top = wrong[P[wrong].max(1).values.argsort(descending=True)[:12]]
fig, axs = plt.subplots(2, 6, figsize=(15, 5.5))
for a, i in zip(axs.ravel(), top):
    a.imshow((X[i].permute(1, 2, 0) * torch.tensor(std) + torch.tensor(mean)).clamp(0, 1))
    a.set_title(f"{classes[Y[i]]} -> {classes[pred[i]]} ({P[i].max():.2f})", fontsize=8); a.axis("off")
plt.suptitle("most confident mistakes"); plt.show()

## 7. 데모 앱 (선택)

In [ ]:
!pip -q install gradio
import gradio as gr
def predict(img):
    x = eval_tf(img).unsqueeze(0).to(device)
    with torch.no_grad(): p = F.softmax(model.eval()(x), -1)[0]
    return {c: float(p[i]) for i, c in enumerate(classes)}
gr.Interface(fn=predict, inputs=gr.Image(type="pil"), outputs=gr.Label(num_top_classes=3)).launch(share=True)

## 8. 보고서 (README) 초안
1. **문제**:
2. **데이터**:
3. **방법**:
4. **결과** (베이스라인 사다리 표 + 테스트 점수):
5. **분석** (오류 사례와 원인):
6. **한계와 다음 단계**:
7. **재현 방법**:

수고 많았습니다. 🎉